# Team Structures Demo

Öffentliche Demo für Teamstrukturen auf Basis von `capitalmarket.capitalselector`.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/gap-labs/meta-credit-dynamics/blob/main/notebooks/team_demo.ipynb)

1. In Colab: `Runtime -> Run all` (Auto-Run per URL wird von Colab nicht erlaubt).
2. Lokal: Repository-Pfad in der Setup-Zelle setzen (`PROJECT_IMPORT_ROOT`).
3. Kernel auswählen und Setup-Zelle ausführen.

In [ ]:
import os, sys, subprocess
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

IN_COLAB = "google.colab" in sys.modules
PUBLIC_REPO_URL = "https://github.com/gap-labs/meta-credit-dynamics.git"
COLAB_REPO_DIR = Path("/content/meta-credit-dynamics")

if IN_COLAB:
    if not COLAB_REPO_DIR.exists():
        subprocess.run(["git", "clone", "--depth", "1", PUBLIC_REPO_URL, str(COLAB_REPO_DIR)], check=True)
    req_file = COLAB_REPO_DIR / "requirements.txt"
    if req_file.exists():
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(req_file)], check=True)
    PROJECT_IMPORT_ROOT = str(COLAB_REPO_DIR)
else:
    # set this to your extracted repo root (where "capitalmarket/" lives)
    PROJECT_IMPORT_ROOT = "/home/andreas/prj/dl"

if PROJECT_IMPORT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_IMPORT_ROOT)

from capitalmarket.capitalselector.builder import CapitalSelectorBuilder
from capitalmarket.capitalselector.population_manager import PopulationManager, RebirthConfig
from capitalmarket.capitalselector.determinism import enable_determinism
from capitalmarket.capitalselector.worlds import GovernanceWorld

## GovernanceWorld project simulation

Die nächste Zelle definiert `simulate_project(...)` und simuliert eine Population über mehrere Schritte.

Ergebnisartefakte:
- globale Governance-Dynamik: `V`, `M`, `K`, `dV`, `dK`
- Populationsmetriken: `wealth`, `wealth_mean_t`, `omega_hist`
- Lifecycle-Signal: `rebirth_counts`

In [ ]:
REGIMES = {
    "Aligned": dict(alpha=0.70, manipulability=0.10, reweight_speed=0.02, punishment=0.10, volatility=0.05),
    "Drift (Goodhart)": dict(alpha=0.40, manipulability=0.55, reweight_speed=0.20, punishment=0.20, volatility=0.10),
    "Punitive (Blame)": dict(alpha=0.60, manipulability=0.25, reweight_speed=0.08, punishment=0.90, volatility=0.10),
    "Volatile (Metrics)": dict(alpha=0.60, manipulability=0.30, reweight_speed=0.30, punishment=0.25, volatility=0.60),
}

def corr(a: np.ndarray, b: np.ndarray) -> float:
    a = np.asarray(a, dtype=float)
    b = np.asarray(b, dtype=float)
    if len(a) < 2 or np.std(a) == 0 or np.std(b) == 0:
        return float("nan")
    return float(np.corrcoef(a, b)[0, 1])

def alignment_index(V: np.ndarray, K: np.ndarray) -> float:
    return corr(V, K)

def gaming_ratio(dK: np.ndarray, dV: np.ndarray) -> float:
    dK = np.asarray(dK)
    dV = np.asarray(dV)
    pos = dK > 0
    if pos.sum() == 0:
        return 0.0
    return float(((dK > 0) & (dV <= 0)).sum() / pos.sum())

def gini(x: np.ndarray) -> float:
    x = np.asarray(x, dtype=float)
    if np.allclose(x.sum(), 0):
        return 0.0
    x = np.sort(np.maximum(x, 0))
    n = x.size
    idx = np.arange(1, n + 1)
    return float((2 * np.sum(idx * x) / (n * np.sum(x))) - (n + 1) / n)

def governance_volatility(omega_hist: np.ndarray) -> float:
    omega_hist = np.asarray(omega_hist, dtype=float)
    if len(omega_hist) < 2:
        return 0.0
    diffs = omega_hist[1:] - omega_hist[:-1]
    return float(np.mean(np.linalg.norm(diffs, axis=1)))

def structural_churn(rebirth_counts: np.ndarray, T: int) -> float:
    return float(np.sum(rebirth_counts) / max(T, 1))

def index_report(V, K, dK, dV, wealth, omega_hist, rebirth_counts):
    return {
        "Alignment Index (AI)": alignment_index(V, K),
        "Gaming Ratio (GR)": gaming_ratio(dK, dV),
        "Concentration Index (CI, Gini)": gini(wealth),
        "Governance Volatility (GV)": governance_volatility(omega_hist),
        "Structural Churn (SC)": structural_churn(rebirth_counts, len(V)),
    }

def simulate_project(regime: dict, *, N=120, T=2000, K_channels=8, seed=42):
    enable_determinism(seed)

    # initial population: N processes with selectors
    processes = {}
    for pid in range(N):
        sel = CapitalSelectorBuilder().with_K(K_channels).build()
        processes[int(pid)] = sel

    manager = PopulationManager(
        processes=processes,
        rebirth_config=RebirthConfig(
            enabled=True,
            base_liquidity=0.20,  # keeps rebirth “alive”
            eta=0.10,
            kappa=1.0,
            max_claims_per_process=1_000_000,
        ),
        backend="cpu",
    )

    world = GovernanceWorld(K_channels=K_channels, regime=regime, seed=seed)

    # histories for indices
    omega_hist = []
    wealth_hist = []
    rebirth_counts = np.zeros(T, dtype=int)

    for t in range(T):
        active_pids = sorted([pid for pid, sel in manager.processes.items() if not bool(getattr(sel, "dead", False))])

        events = world.step(t, active_pids)
        # PopulationManager expects process_events mapping: pid -> {"r_vec","c_total","freeze"}
        out = manager.step_tau(tau=int(t), process_events=events, jackpot=0.0)

        rebirth_counts[t] = len(out.get("newborn_ids", []))

        # sample system state: mean wealth + mean weights over living
        living = [manager.processes[pid] for pid in active_pids]
        if living:
            wealth_hist.append(float(np.mean([float(s.wealth) for s in living])))
            omega_hist.append(np.mean([s.w for s in living if getattr(s, "w", None) is not None], axis=0))
        else:
            wealth_hist.append(float("nan"))
            omega_hist.append(np.full(K_channels, np.nan))

    omega_hist = np.asarray(omega_hist, dtype=float)
    wealth_final = np.array([float(s.wealth) for s in manager.processes.values()], dtype=float)

    V = np.asarray(world.V_hist, dtype=float)
    M = np.asarray(world.M_hist, dtype=float)
    K = np.asarray(world.K_hist, dtype=float)

    dV = np.diff(np.r_[0.0, V])
    dK = np.diff(np.r_[0.0, K])

    return {
        "V": V, "M": M, "K": K,
        "dV": dV, "dK": dK,
        "wealth": wealth_final,
        "omega_hist": omega_hist,
        "rebirth_counts": rebirth_counts,
        "wealth_mean_t": np.asarray(wealth_hist, dtype=float),
    }

In [ ]:
REGIME_NAME = "Drift (Goodhart)"
regime = REGIMES[REGIME_NAME]

sim = simulate_project(regime, N=120, T=2000, K_channels=8, seed=42)

report = index_report(sim["V"], sim["K"], sim["dK"], sim["dV"], sim["wealth"], sim["omega_hist"], sim["rebirth_counts"])
pd.Series(report).to_frame("value")

In [ ]:
plt.figure()
plt.plot(sim["V"], label="True Value V(t)")
plt.plot(sim["K"], label="Rewarded KPI K(t)")
plt.legend()
plt.title(f"{REGIME_NAME}: Value vs KPI")
plt.show()

plt.figure()
plt.plot(sim["wealth_mean_t"], label="Mean wealth (living)")
plt.legend()
plt.title(f"{REGIME_NAME}: Mean wealth over time")
plt.show()

plt.figure()
plt.plot(np.cumsum(sim["rebirth_counts"]), label="Cumulative rebirths")
plt.legend()
plt.title(f"{REGIME_NAME}: Structural churn")
plt.show()

In [ ]:
RUN_CFG = dict(N=120, T=2000, K_channels=8, seed=42)

all_sims = {}
all_reports = {}

for regime_name, regime_cfg in REGIMES.items():
    sim_i = simulate_project(regime_cfg, **RUN_CFG)
    report_i = index_report(
        sim_i["V"], sim_i["K"], sim_i["dK"], sim_i["dV"],
        sim_i["wealth"], sim_i["omega_hist"], sim_i["rebirth_counts"]
    )
    all_sims[regime_name] = sim_i
    all_reports[regime_name] = report_i

metric_order = [
    "Alignment Index (AI)",
    "Gaming Ratio (GR)",
    "Concentration Index (CI, Gini)",
    "Governance Volatility (GV)",
    "Structural Churn (SC)",
]

metrics_df = pd.DataFrame(all_reports).T[metric_order]
metrics_df

fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# Vergleich 1: AI vs GR (Bubblegröße = Structural Churn)
ax = axes[0]
for regime_name, row in metrics_df.iterrows():
    churn = float(row["Structural Churn (SC)"])
    ax.scatter(
        float(row["Alignment Index (AI)"]),
        float(row["Gaming Ratio (GR)"]),
        s=400 + 12000 * churn,
        alpha=0.7,
    )
    ax.annotate(regime_name, (float(row["Alignment Index (AI)"]), float(row["Gaming Ratio (GR)"])), xytext=(6, 4), textcoords="offset points")

ax.set_title("Regime comparison: Alignment vs Gaming")
ax.set_xlabel("Alignment Index (higher is better)")
ax.set_ylabel("Gaming Ratio (lower is better)")
ax.grid(alpha=0.25)

# Vergleich 2: KPI-Value-Gap über Zeit (K - V)
ax = axes[1]
for regime_name, sim_i in all_sims.items():
    gap = np.asarray(sim_i["K"], dtype=float) - np.asarray(sim_i["V"], dtype=float)
    ax.plot(gap, label=regime_name, linewidth=1.4)

ax.axhline(0.0, color="black", linewidth=1.0, alpha=0.6)
ax.set_title("KPI-Value gap over time")
ax.set_xlabel("t")
ax.set_ylabel("K(t) - V(t)")
ax.legend(fontsize=8)
ax.grid(alpha=0.25)

fig.tight_layout()
plt.show()

In [ ]:
# Gesamt-Score über alle Regimes (0..1, höher = besser)
score_cfg = {
    "Alignment Index (AI)": dict(higher_is_better=True, weight=1.0),
    "Gaming Ratio (GR)": dict(higher_is_better=False, weight=1.0),
    "Concentration Index (CI, Gini)": dict(higher_is_better=False, weight=1.0),
    "Governance Volatility (GV)": dict(higher_is_better=False, weight=1.0),
    "Structural Churn (SC)": dict(higher_is_better=False, weight=1.0),
}

norm_parts = {}
for metric, cfg in score_cfg.items():
    col = metrics_df[metric].astype(float)
    min_v, max_v = float(col.min()), float(col.max())
    if np.isclose(max_v, min_v):
        n = pd.Series(0.5, index=col.index, dtype=float)
    else:
        if cfg["higher_is_better"]:
            n = (col - min_v) / (max_v - min_v)
        else:
            n = (max_v - col) / (max_v - min_v)
    norm_parts[metric] = n

norm_df = pd.DataFrame(norm_parts)
weights = pd.Series({k: float(v["weight"]) for k, v in score_cfg.items()})
weights = weights / weights.sum()

score_df = norm_df.copy()
score_df["Overall Score"] = (norm_df * weights).sum(axis=1)
score_df = score_df.sort_values("Overall Score", ascending=False)

display(score_df.round(3))

plt.figure(figsize=(8, 4.8))
bars = plt.barh(score_df.index, score_df["Overall Score"], alpha=0.8)
plt.gca().invert_yaxis()
plt.xlim(0, 1)
plt.xlabel("Overall Score (0..1)")
plt.title("Regime ranking (normalized composite score)")
plt.grid(axis="x", alpha=0.25)

for bar, value in zip(bars, score_df["Overall Score"]):
    plt.text(value + 0.01, bar.get_y() + bar.get_height() / 2, f"{value:.3f}", va="center")

plt.tight_layout()
plt.show()

In [ ]:
# Gewichtete Variante + Robustheitscheck über mehrere Seeds
WEIGHTED_SCORE_CFG = {
    "Alignment Index (AI)": dict(higher_is_better=True, weight=2.0),
    "Gaming Ratio (GR)": dict(higher_is_better=False, weight=1.5),
    "Concentration Index (CI, Gini)": dict(higher_is_better=False, weight=1.0),
    "Governance Volatility (GV)": dict(higher_is_better=False, weight=1.0),
    "Structural Churn (SC)": dict(higher_is_better=False, weight=0.5),
}

def compute_composite_scores(metrics_table: pd.DataFrame, score_cfg: dict) -> pd.DataFrame:
    norm_parts_local = {}
    for metric, cfg in score_cfg.items():
        col = metrics_table[metric].astype(float)
        min_v, max_v = float(col.min()), float(col.max())
        if np.isclose(max_v, min_v):
            n = pd.Series(0.5, index=col.index, dtype=float)
        elif cfg["higher_is_better"]:
            n = (col - min_v) / (max_v - min_v)
        else:
            n = (max_v - col) / (max_v - min_v)
        norm_parts_local[metric] = n

    norm_df_local = pd.DataFrame(norm_parts_local)
    weights_local = pd.Series({k: float(v["weight"]) for k, v in score_cfg.items()})
    weights_local = weights_local / weights_local.sum()

    out = norm_df_local.copy()
    out["Overall Score"] = (norm_df_local * weights_local).sum(axis=1)
    return out.sort_values("Overall Score", ascending=False)

# 1) Gewichtetes Ranking für den aktuellen (Seed=42) Multi-Regime-Run
weighted_score_df = compute_composite_scores(metrics_df, WEIGHTED_SCORE_CFG)
display(weighted_score_df.round(3))

# 2) Robustheit: Ranking über mehrere Seeds
ROBUST_SEEDS = [11, 19, 23, 29, 31]
ROBUST_RUN_CFG = dict(RUN_CFG)
ROBUST_RUN_CFG["T"] = 1000  # etwas kürzer für schnelleren Mehrfachlauf

robust_rows = []
for seed_i in ROBUST_SEEDS:
    seed_reports = {}
    for regime_name, regime_cfg in REGIMES.items():
        sim_i = simulate_project(regime_cfg, seed=seed_i, N=ROBUST_RUN_CFG["N"], T=ROBUST_RUN_CFG["T"], K_channels=ROBUST_RUN_CFG["K_channels"])
        seed_reports[regime_name] = index_report(
            sim_i["V"], sim_i["K"], sim_i["dK"], sim_i["dV"],
            sim_i["wealth"], sim_i["omega_hist"], sim_i["rebirth_counts"]
        )

    seed_metrics = pd.DataFrame(seed_reports).T[list(WEIGHTED_SCORE_CFG.keys())]
    seed_scores = compute_composite_scores(seed_metrics, WEIGHTED_SCORE_CFG)
    for regime_name, row in seed_scores.iterrows():
        robust_rows.append({
            "seed": int(seed_i),
            "Regime": regime_name,
            "Overall Score": float(row["Overall Score"]),
        })

robust_df = pd.DataFrame(robust_rows)
robust_summary = robust_df.groupby("Regime")["Overall Score"].agg(["mean", "std", "min", "max"]).sort_values("mean", ascending=False)
display(robust_summary.round(3))

order = robust_summary.index.tolist()

fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# Vergleich A: Mittelwert +/- Std je Regime
ax = axes[0]
means = robust_summary["mean"].to_numpy()
stds = robust_summary["std"].fillna(0.0).to_numpy()
x = np.arange(len(order))
ax.bar(x, means, yerr=stds, capsize=4, alpha=0.8)
ax.set_ylim(0, 1)
ax.set_title("Weighted score robustness: mean ± std")
ax.set_ylabel("Overall Score")
ax.set_xticks(x)
ax.set_xticklabels(order, rotation=20, ha="right")
ax.grid(axis="y", alpha=0.25)

# Vergleich B: Verteilung je Regime über Seeds
ax = axes[1]
box_data = [robust_df.loc[robust_df["Regime"] == r, "Overall Score"].to_numpy() for r in order]
try:
    ax.boxplot(box_data, tick_labels=order)
except TypeError:
    ax.boxplot(box_data, labels=order)
ax.set_ylim(0, 1)
ax.set_title("Weighted score distribution across seeds")
ax.set_ylabel("Overall Score")
ax.set_xticklabels(order, rotation=20, ha="right")
ax.grid(axis="y", alpha=0.25)

fig.tight_layout()
plt.show()

In [ ]:
# Sensitivitätsanalyse: Wie stark hängt das Ranking von der Gewichtung ab?
WEIGHT_SCHEMES = {
    "Balanced": {
        "Alignment Index (AI)": dict(higher_is_better=True, weight=1.0),
        "Gaming Ratio (GR)": dict(higher_is_better=False, weight=1.0),
        "Concentration Index (CI, Gini)": dict(higher_is_better=False, weight=1.0),
        "Governance Volatility (GV)": dict(higher_is_better=False, weight=1.0),
        "Structural Churn (SC)": dict(higher_is_better=False, weight=1.0),
    },
    "Alignment-First": {
        "Alignment Index (AI)": dict(higher_is_better=True, weight=3.0),
        "Gaming Ratio (GR)": dict(higher_is_better=False, weight=1.5),
        "Concentration Index (CI, Gini)": dict(higher_is_better=False, weight=0.8),
        "Governance Volatility (GV)": dict(higher_is_better=False, weight=0.8),
        "Structural Churn (SC)": dict(higher_is_better=False, weight=0.4),
    },
    "Stability-First": {
        "Alignment Index (AI)": dict(higher_is_better=True, weight=1.5),
        "Gaming Ratio (GR)": dict(higher_is_better=False, weight=1.0),
        "Concentration Index (CI, Gini)": dict(higher_is_better=False, weight=1.0),
        "Governance Volatility (GV)": dict(higher_is_better=False, weight=2.0),
        "Structural Churn (SC)": dict(higher_is_better=False, weight=2.0),
    },
}

sens_rows = []
for scheme_name, scheme_cfg in WEIGHT_SCHEMES.items():
    scheme_scores = compute_composite_scores(metrics_df, scheme_cfg)
    for rank_i, (regime_name, row) in enumerate(scheme_scores.iterrows(), start=1):
        sens_rows.append({
            "Scheme": scheme_name,
            "Regime": regime_name,
            "Overall Score": float(row["Overall Score"]),
            "Rank": int(rank_i),
        })

sens_df = pd.DataFrame(sens_rows)
score_pivot = sens_df.pivot(index="Regime", columns="Scheme", values="Overall Score")
rank_pivot = sens_df.pivot(index="Regime", columns="Scheme", values="Rank")

sens_summary = pd.DataFrame({
    "score_mean": score_pivot.mean(axis=1),
    "score_range": score_pivot.max(axis=1) - score_pivot.min(axis=1),
    "rank_mean": rank_pivot.mean(axis=1),
    "rank_range": rank_pivot.max(axis=1) - rank_pivot.min(axis=1),
}).sort_values(["rank_mean", "score_mean"], ascending=[True, False])

print("Scores by scheme")
display(score_pivot.round(3))
print("Ranks by scheme (1 = best)")
display(rank_pivot.astype(int))
print("Sensitivity summary")
display(sens_summary.round(3))

regime_order = sens_summary.index.tolist()
scheme_order = list(WEIGHT_SCHEMES.keys())

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# Plot A: Gruppierte Balken (Scores je Schema)
ax = axes[0]
x = np.arange(len(regime_order))
width = 0.22
for j, scheme_name in enumerate(scheme_order):
    vals = score_pivot.loc[regime_order, scheme_name].to_numpy()
    ax.bar(x + (j - (len(scheme_order) - 1) / 2) * width, vals, width=width, label=scheme_name, alpha=0.85)

ax.set_title("Score sensitivity across weighting schemes")
ax.set_ylabel("Overall Score")
ax.set_ylim(0, 1)
ax.set_xticks(x)
ax.set_xticklabels(regime_order, rotation=20, ha="right")
ax.grid(axis="y", alpha=0.25)
ax.legend(fontsize=8)

# Plot B: Rangprofil je Regime
ax = axes[1]
x2 = np.arange(len(scheme_order))
for regime_name in regime_order:
    y = rank_pivot.loc[regime_name, scheme_order].to_numpy(dtype=float)
    ax.plot(x2, y, marker="o", linewidth=1.8, label=regime_name)

ax.set_title("Rank profile across weighting schemes")
ax.set_ylabel("Rank (1 = best)")
ax.set_xticks(x2)
ax.set_xticklabels(scheme_order, rotation=15, ha="right")
ax.set_yticks(np.arange(1, len(regime_order) + 1))
ax.invert_yaxis()
ax.grid(alpha=0.25)
ax.legend(fontsize=8)

fig.tight_layout()
plt.show()

In [ ]:
# Konsens-Ranking über Gewichtungsschemata (Borda + Mean Rank)
if "rank_pivot" not in globals() or "score_pivot" not in globals():
    raise RuntimeError("Bitte zuerst Zelle 10 (Sensitivitätsanalyse) ausführen.")

n_regimes = rank_pivot.shape[0]
borda_points = (n_regimes + 1 - rank_pivot).sum(axis=1)
mean_rank = rank_pivot.mean(axis=1)
rank_std = rank_pivot.std(axis=1)
score_mean = score_pivot.mean(axis=1)
score_std = score_pivot.std(axis=1)

consensus_df = pd.DataFrame({
    "Borda Points": borda_points,
    "Mean Rank": mean_rank,
    "Rank Std": rank_std,
    "Mean Score": score_mean,
    "Score Std": score_std,
    "Best Rank": rank_pivot.min(axis=1),
    "Worst Rank": rank_pivot.max(axis=1),
})

consensus_df = consensus_df.sort_values(["Borda Points", "Mean Score"], ascending=[False, False])
consensus_df["Consensus Rank"] = np.arange(1, len(consensus_df) + 1)

def classify_regime(row: pd.Series) -> str:
    if row["Consensus Rank"] == 1 and row["Worst Rank"] <= 2:
        return "Robust Top"
    if row["Consensus Rank"] <= 2 and row["Worst Rank"] > 2:
        return "Contested Top"
    if row["Best Rank"] == row["Worst Rank"]:
        return "Stable Position"
    return "Weight-Sensitive"

consensus_df["Interpretation"] = consensus_df.apply(classify_regime, axis=1)

display_cols = [
    "Consensus Rank", "Interpretation", "Borda Points",
    "Mean Rank", "Rank Std", "Mean Score", "Score Std", "Best Rank", "Worst Rank"
 ]
display(consensus_df[display_cols].round(3))

plt.figure(figsize=(8.2, 4.8))
plot_df = consensus_df.sort_values("Consensus Rank")
bars = plt.barh(plot_df.index, plot_df["Borda Points"], alpha=0.85)
plt.gca().invert_yaxis()
plt.title("Consensus ranking by Borda points")
plt.xlabel("Borda Points (higher = better)")
plt.grid(axis="x", alpha=0.25)

for bar, val in zip(bars, plot_df["Borda Points"]):
    plt.text(bar.get_width() + 0.05, bar.get_y() + bar.get_height()/2, f"{val:.1f}", va="center")

plt.tight_layout()
plt.show()

## Executive Summary (auto-generated)

Die Zusammenfassung wird in der nächsten Code-Zelle automatisch aus den aktuellen Analyse-DataFrames erzeugt.

In [ ]:
from IPython.display import Markdown, display

required = ["consensus_df", "robust_summary", "sens_summary", "rank_pivot", "score_pivot"]
missing = [name for name in required if name not in globals()]
if missing:
    raise RuntimeError(
        "Bitte zuerst Zellen 7 bis 11 ausführen, damit alle Auswertungen für die Summary vorliegen. "
        f"Fehlend: {', '.join(missing)}"
    )

cons_sorted = consensus_df.sort_values("Consensus Rank")
top_1 = cons_sorted.index[0]
top_2 = cons_sorted.index[1]

robust_top = robust_summary["mean"].idxmax()
robust_bottom = robust_summary["mean"].idxmin()
robust_top_mean = float(robust_summary.loc[robust_top, "mean"])
robust_bottom_mean = float(robust_summary.loc[robust_bottom, "mean"])

most_score_sensitive = sens_summary["score_range"].idxmax()
score_range_max = float(sens_summary.loc[most_score_sensitive, "score_range"])

rank_sensitive = sens_summary[sens_summary["rank_range"] > 0].sort_values("rank_range", ascending=False).index.tolist()
if rank_sensitive:
    rank_sensitive_txt = ", ".join(rank_sensitive)
else:
    rank_sensitive_txt = "keines (alle Ränge stabil)"

borda_txt = ", ".join([
    f"{int(row['Consensus Rank'])}) {regime} ({int(row['Borda Points'])})"
    for regime, row in cons_sorted.iterrows()
])

md = f"""
## Executive Summary

- Konsens-Ranking (Borda): **{borda_txt}**.
- Die Top-Entscheidung bleibt zwischen **{top_1}** und **{top_2}**.
- Im Multi-Seed-Check führt **{robust_top}** mit mittlerem Score **{robust_top_mean:.3f}**; Schlusslicht ist **{robust_bottom}** mit **{robust_bottom_mean:.3f}**.
- Höchste Score-Sensitivität gegenüber Gewichten: **{most_score_sensitive}** (Range **{score_range_max:.3f}**).
- Rangsensitiv über Schemata: **{rank_sensitive_txt}**.
"""
display(Markdown(md))